<font size=10>**DATA EXPLORATION & PREPROCESSING**</font> <a class="anchor" id='title'></a> 

**Bachelor's in Data Science - NOVA IMS (25/26)**

<font color='#BFD72' size=5>**RESEARCH QUESTION**: </font><font size=5>*Which companies have a dominant position in public procurement?*</font> 

**Data**: 
- [*Portal BASE*](https://www.base.gov.pt/Base4/pt/pesquisa/?type=contratos&texto=&adjudicante=&adjudicataria=&tipo=2&tipocontrato=0&cpv=&aqinfo=&desdeprazoexecucao=&ateprazoexecucao=&sel_price=price_11&desdeprecocontrato=&ateprecocontrato=&desdeprecoefectivo=&ateprecoefectivo=&sel_date=date_11&desdedatacontrato=2023-01-01&atedatacontrato=2026-03-31&desdedatapublicacao=&atedatapublicacao=&desdedatafecho=&atedatafecho=&pais=0&distrito=0&concelho=0)

- [*Treated Datasets*](https://dados.gov.pt/pt/datasets/contratos-publicos-portal-base-impic-contratos-de-2012-a-2026/#/resources)

**Group B**
- Beatriz Marques 20231605
- Maria Inês Santos 20231630
- Luís Soeiro 20211536
- Rodrigo Silva 20231602

<font color='#BFD72' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>  
- [1. Imports](#1-imports)  
- [2. Data Integration](#2-data-integration)  
- [3. Data Preprocessing](#3-data-preprocessing)  
    - [3.1 Duplicates](#31-duplicates)
    - [3.2 Missing Values](#32-missing-values)
    - [3.3 Preprocessing Per Column](#33-preprocessing-per-column)
        - [3.3.1 Date Columns](#331-date-columns)
        - [3.3.2 Tipo de Procedimento](#332-tipo-de-procedimento)
        - [3.3.3 Tipo de Contrato](#333-tipo-de-contrato)
        - [3.3.4 Concorrentes](#334-concorrentes)
        - [3.3.5 Local de Execução](#335-local-de-execução)
        - [3.3.6 Price Columns](#336-price-columns)
        - [3.3.7 CPV](#337-cpv)
        - [3.3.8 Adjudicante & Adjudicatário](#338-adjudicante--adjudicatário)
    - [3.4 Final Data](#34-final-data)
- [4. Export Preprocessed Data](#4-export-preprocessed-data)

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import subprocess, sys, importlib
import pandas as pd
import plotly.express as px
import re
import plotly.graph_objects as go
import numpy as np
import unicodedata
import re

import warnings
warnings.filterwarnings('ignore')

try:
    import openpyxl
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
    importlib.invalidate_caches()
    import openpyxl
import os

In [3]:
# Get the absolute path of the source_code folder
source_code_path = os.path.abspath('../source')

# Add the source_code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

# <font color='#BFD72F' size=6>**2. Data Integration**</font> <a class="anchor" id="2"></a>
  
[Back to TOC](#toc)

$\rightarrow$ **Contract Timeframe**: From January 1, 2023 to April 25, 2026

In [4]:
# MERGE DATASETS
paths = {
    "2023_part01": "../data/contratos2023_part01.csv",
    "2023_part02": "../data/contratos2023_part02.csv",
    "2023_part03": "../data/contratos2023_part03.csv",
    "2024_part01": "../data/contratos2024_part01.csv",
    "2024_part02": "../data/contratos2024_part02.csv",
    "2024_part03": "../data/contratos2024_part03.csv",
    "2025_part01": "../data/contratos2025_part01.csv",
    "2025_part02": "../data/contratos2025_part02.csv",
    "2025_part03": "../data/contratos2025_part03.csv",
    "2026": "../data/contratos2026.csv",
}

datasets = {}

merged_dataset = pd.DataFrame()

for year, path in paths.items():
    print(f"Loading dataset for {year} from {path}...")
    datasets[year] = pd.read_csv(path)
    print(f"Dataset for {year} loaded successfully with shape {datasets[year].shape}.")
    merged_dataset = pd.concat([merged_dataset, datasets[year]], ignore_index=True)

print(f"Merged dataset created with shape {merged_dataset.shape}.")

Loading dataset for 2023_part01 from ../data/contratos2023_part01.csv...
Dataset for 2023_part01 loaded successfully with shape (64562, 35).
Loading dataset for 2023_part02 from ../data/contratos2023_part02.csv...
Dataset for 2023_part02 loaded successfully with shape (64562, 35).
Loading dataset for 2023_part03 from ../data/contratos2023_part03.csv...
Dataset for 2023_part03 loaded successfully with shape (64562, 35).
Loading dataset for 2024_part01 from ../data/contratos2024_part01.csv...
Dataset for 2024_part01 loaded successfully with shape (75440, 35).
Loading dataset for 2024_part02 from ../data/contratos2024_part02.csv...
Dataset for 2024_part02 loaded successfully with shape (75440, 35).
Loading dataset for 2024_part03 from ../data/contratos2024_part03.csv...
Dataset for 2024_part03 loaded successfully with shape (75441, 35).
Loading dataset for 2025_part01 from ../data/contratos2025_part01.csv...
Dataset for 2025_part01 loaded successfully with shape (81131, 35).
Loading datas

In [5]:
merged_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 732573 entries, 0 to 732572
Data columns (total 35 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   idcontrato                732573 non-null  int64  
 1   nAnuncio                  114355 non-null  str    
 2   TipoAnuncio               114355 non-null  str    
 3   idINCM                    114355 non-null  float64
 4   tipoContrato              732572 non-null  str    
 5   idprocedimento            732573 non-null  int64  
 6   tipoprocedimento          732573 non-null  str    
 7   objectoContrato           732571 non-null  str    
 8   descContrato              732573 non-null  str    
 9   adjudicante               732565 non-null  str    
 10  adjudicatarios            731904 non-null  str    
 11  dataPublicacao            732573 non-null  str    
 12  dataCelebracaoContrato    730239 non-null  str    
 13  precoContratual           732573 non-null  float64
 14 

**COLUMNS TO KEEP**

| Column Name        | Description                                                                                    | Examples                                                                                                                                      |
| ------------------ | ---------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------- |
| `idcontrato`       | Unique identifier assigned to each public procurement contract.                                | `11103352`, `11104095`, `11122539`                                                                                                            |
| `tipoContrato`     | Type/category of the contract being awarded.                                                   | `Empreitadas de obras públicas`, `Aquisição de serviços`, `Aquisição de bens móveis`                                                          |
| `tipoFimContrato`  | Type of contract termination or completion status.                                             | `Resolução`, `Caducidade`, `Conclusão do contrato`                                                                                            |
| `CPV`              | Common Procurement Vocabulary (CPV) code and description identifying the procurement category. | `45233120-6 - Construção de estradas`, `50241000-6 - Serviços de reparação e manutenção`, `50750000-7 - Serviços de manutenção de elevadores` |
| `tipoprocedimento` | Procurement procedure used for awarding the contract.                                          | `Concurso público`, `Ajuste Direto Regime Geral`, `Consulta Prévia`                                                                           |


<br>


| Column Name      | Description                                                                     | Examples                                                                            |
| ---------------- | ------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------- |
| `adjudicante`    | Contracting authority or public entity responsible for the procurement process. | `Infraestruturas de Portugal, S. A.`, `Município de Nelas`, `Universidade do Porto` |
| `adjudicatarios` | Company or entity awarded the contract.                                         | `CJR`, `AGR - Engenharia e Serviços, Lda`, `TK Elevadores, Unipessoal Lda`          |
| `concorrentes`   | Companies/entities that submitted bids in the procurement procedure.            | `Conduril – Engenharia, S.A.`, `AGR - Engenharia e Serviços, Lda`, `Mathias`        |


<br>


| Column Name             | Description                                                                      | Examples                               |
| ----------------------- | -------------------------------------------------------------------------------- | -------------------------------------- |
| `precoBaseProcedimento` | Initial maximum budget/value defined for the procurement procedure.              | `18000000.00`, `979644.72`, `55020.00` |
| `precoContratual`       | Final contractual value agreed between contracting authority and awarded entity. | `14665084.64`, `10776.00`, `52511.52`  |
| `PrecoTotalEfetivo`     | Total effective amount actually paid/executed during the contract lifecycle.     | `0.0`, `152000.50`, `98500.00`         |

<br>


| Column Name     | Description                                                 | Examples                                                                        |
| --------------- | ----------------------------------------------------------- | ------------------------------------------------------------------------------- |
| `LocalExecucao` | Geographic location where the contract/project is executed. | `Portugal, Porto, Baião`, `Portugal, Setúbal, Almada`, `Portugal, Viseu, Nelas` |


<br>


| Column Name              | Description                                                  | Examples                                 |
| ------------------------ | ------------------------------------------------------------ | ---------------------------------------- |
| `dataDecisaoAdjudicacao` | Date on which the award decision was officially made.        | `2024-11-08`, `2024-12-20`, `2024-12-30` |
| `dataCelebracaoContrato` | Date on which the contract was formally signed/celebrated.   | `2025-04-24`, `2025-01-14`, `2025-01-02` |
| `dataPublicacao`         | Date on which the contract/procedure was publicly published. | `2024-12-23`, `2024-12-24`, `2025-01-02` |
| `dataFechoContrato`      | Date on which the contract was closed/completed.             | `2025-06-30`, `2024-12-31`, `2025-03-15` |

In [6]:
cols_to_keep = [
    'idcontrato', 'tipoContrato', 'tipoFimContrato', 'CPV', 'tipoprocedimento',
    'adjudicante', 'adjudicatarios', 'concorrentes', 
    'precoBaseProcedimento', 'precoContratual', 'PrecoTotalEfetivo', 
    'LocalExecucao', 
    "dataDecisaoAdjudicacao", "dataCelebracaoContrato", "dataPublicacao", "dataFechoContrato"
]

subset = merged_dataset[cols_to_keep]

In [7]:
print("There are {} Public Entities.".format(subset['adjudicante'].nunique()))
print("There are {} Companies.".format(subset['adjudicatarios'].nunique()))

print("So, in total our analysis contains {} Nodes.".format(
    subset['adjudicatarios'].nunique() + 
    subset['adjudicante'].nunique()))

There are 10309 Public Entities.
There are 143783 Companies.
So, in total our analysis contains 154092 Nodes.


# <font color='#BFD72F' size=6>**3. Data Preprocessing**</font> <a class="anchor" id="3"></a>
  
[Back to TOC](#toc)

## <font size=6>**3.1 Duplicates**</font> <a class="anchor" id="3.1"></a>
  
[Back to TOC](#toc)

In [8]:
# Check duplicated rows
duplicated_rows = subset.duplicated()
print(f"Number of duplicated rows: {duplicated_rows.sum()}")

Number of duplicated rows: 2826


In [9]:
# dropping duplicated rows
subset = subset.drop_duplicates()

## <font size=6>**3.2 Missing Values**</font> <a class="anchor" id="3.2"></a>
  
[Back to TOC](#toc)

In [10]:
total = len(subset)
missing_counts = subset.isnull().sum()
missing_pct = (missing_counts / total * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_pct': missing_pct
}).sort_values('missing_pct', ascending=False)

# show only columns with any missing values
missing_df

,missing_count,missing_pct
tipoFimContrato,579308,79.38
dataFechoContrato,576960,79.06
concorrentes,388427,53.23
dataDecisaoAdjudicacao,2334,0.32
dataCelebracaoContrato,2334,0.32
LocalExecucao,1647,0.23
adjudicatarios,658,0.09
idcontrato,0,0.00
tipoprocedimento,0,0.00
adjudicante,8,0.00


In [11]:
# rows with missing values except for 'concorrentes', 'dataFechoContrato' and 'tipoFimContrato'
subset = subset.dropna(subset=[col for col in subset.columns if col not in ['concorrentes', 'dataFechoContrato', 'tipoFimContrato']])

In [12]:
print("Initial Number of Contracts: {}".format(merged_dataset.shape[0]))
print("Number of Contracts Now: {}".format(subset.shape[0]))
print("Percentage of Deleted Contracts: {}%".format(round((1 - (subset.shape[0]/merged_dataset.shape[0]))*100, 2)))

Initial Number of Contracts: 732573
Number of Contracts Now: 726745
Percentage of Deleted Contracts: 0.8%


## <font size=6>**3.3 Preprocessing Per Column**</font> <a class="anchor" id="3.3"></a>
  
[Back to TOC](#toc)

### <font size=6>3.3.1 Date Columns</font> <a class="anchor" id="3.3.1"></a>
  
[Back to TOC](#toc)

In [13]:
# date type conversion    
subset["dataPublicacao"] = pd.to_datetime(subset["dataPublicacao"], errors='coerce')
subset["dataCelebracaoContrato"] = pd.to_datetime(subset["dataCelebracaoContrato"], errors='coerce')
subset["dataDecisaoAdjudicacao"] = pd.to_datetime(subset["dataDecisaoAdjudicacao"], errors='coerce')
subset["dataFechoContrato"] = pd.to_datetime(subset["dataFechoContrato"], errors='coerce')

In [14]:
dfs = []

date_cols = {
    'dataDecisaoAdjudicacao': 'Decision',
    'dataCelebracaoContrato': 'Celebration',
    'dataFechoContrato': 'Closure'
}

for col, label in date_cols.items():
    df = (
        subset
        .dropna(subset=[col])
        .assign(month=subset[col].dt.to_period('M').dt.to_timestamp())
        .groupby('month')
        .size()
        .reset_index(name='number_of_contracts')
    )
    
    df['type'] = label
    dfs.append(df)

final_df = pd.concat(dfs)

fig = px.line(
    final_df,
    x='month',
    y='number_of_contracts',
    color='type',
    title='Number of Contracts by Month (Different Dates)',
    labels={
        'month': 'Month',
        'number_of_contracts': 'Number of Contracts',
        'type': 'Date Type'
    }
)

fig.show()

In [15]:
# compute difference in months between celebration and closure, then plot histogram
months_df = subset.dropna(subset=['dataCelebracaoContrato','dataFechoContrato']).copy()
s = months_df['dataCelebracaoContrato']
e = months_df['dataFechoContrato']
months_df['months_diff'] = (e.dt.year - s.dt.year) * 12 + (e.dt.month - s.dt.month) + (e.dt.day - s.dt.day) / 30.0

fig_months = px.histogram(
    months_df,
    x='months_diff',
    nbins=100,
    title='Months between Celebration and Closure',
    labels={'months_diff': 'Months difference', 'count': 'Number of Contracts'}
)
fig_months.update_xaxes(range=[float(months_df['months_diff'].min()), float(months_df['months_diff'].max())])
fig_months.show()


In [16]:
# cut where closure date is before celebration date -> inconsistent data that should be removed
subset = subset[subset['dataFechoContrato'] >= subset['dataCelebracaoContrato']]

### <font size=6>3.3.2 Tipo de Procedimento</font> <a class="anchor" id="3.3.2"></a>
  
[Back to TOC](#toc)

$\rightarrow$ **tipoprocedimento**: Concurso público

In [17]:
subset = subset[subset['tipoprocedimento'] == 'Concurso público']
subset.drop(columns=['tipoprocedimento'], inplace=True)

In [18]:
subset.shape

(16994, 15)

### <font size=6>3.3.3 Tipo de Contrato</font> <a class="anchor" id="3.3.3"></a>
  
[Back to TOC](#toc)

In [19]:
subset['tipoContrato'] = subset['tipoContrato'].astype(str).str.replace(r'[\r\n]+', ' | ', regex=True).str.strip()

In [20]:
subset['tipoContrato'].value_counts()

tipoContrato
Aquisição de bens móveis                                                          9342
Aquisição de serviços                                                             4175
Empreitadas de obras públicas                                                     2866
Locação de bens móveis                                                             402
Aquisição de bens móveis | Aquisição de serviços                                   136
Aquisição de serviços | Locação de bens móveis                                      48
Concessão de serviços públicos                                                      12
Aquisição de bens móveis | Locação de bens móveis                                    3
Aquisição de serviços | Empreitadas de obras públicas                                2
Empreitadas de obras públicas | Locação de bens móveis                               2
Aquisição de bens móveis | Empreitadas de obras públicas                             2
Concessão de obras públicas   

In [21]:
subset_counts = subset['tipoContrato'].value_counts().reset_index()
subset_counts.columns = ['tipoContrato', 'count']

# Get top 10
top_10 = subset_counts.head(10)

fig = px.bar(
    top_10,
    x='tipoContrato',
    y='count',
    title='Number of Contracts by Contract Type',
    labels={'tipoContrato': 'Contract Type', 'count': 'Number of Contracts'}
)

fig.show()

### <font size=6>3.3.4 Concorrentes</font> <a class="anchor" id="3.3.4"></a>
  
[Back to TOC](#toc)

In [22]:
subset['concorrentes'] = (
    subset['concorrentes']
    .astype(str)
    # replace line breaks with separator
    .str.replace(r'[\r\n]+', ' | ', regex=True)
    # remove excessive spaces
    .str.replace(r'\s+', ' ', regex=True)
    # remove trailing numbers (like " 102", " 89", etc.)
    .str.replace(r'\s+\d+\s*$', '', regex=True)
    # final trim
    .str.strip()
)

In [23]:
# number of competitors per contract
subset['nr_concorrentes'] = subset['concorrentes'].apply(
    lambda x: len([i for i in re.split(r'\s*\|\s*', x) if i]) 
    if isinstance(x, str) else 0
)

fig = px.histogram(
    subset,
    x='nr_concorrentes',
    nbins=30,
    title='Histogram of Number of Competitors'
)

fig.show()

### <font size=6>3.3.5 Local de Execução</font> <a class="anchor" id="3.3.5"></a>
  
[Back to TOC](#toc)

In [24]:
# LocalExecucao
subset['LocalExecucao'] = subset['LocalExecucao'].fillna('')

subset['LocalExecucao'] = (
    subset['LocalExecucao']
    # replace line breaks with separator
    .str.replace(r'[\r\n]+', ' | ', regex=True)
    # normalize spaces
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    # remove duplicates inside each cell
    .apply(lambda x: ' | '.join(dict.fromkeys(x.split(' | '))) if x else x)
)

first_location = subset['LocalExecucao'].str.split(' \| ', expand=False).str[0]

split_cols = first_location.str.split(', ', expand=True)

split_cols = split_cols.reindex(columns=[0, 1, 2])
split_cols.columns = ['country', 'district', 'city']

subset[['country', 'district', 'city']] = split_cols

for col in ['country', 'district', 'city']:
    subset[col] = subset[col].replace(r'^\s*$', pd.NA, regex=True)

# handle inconsistent structures
n_parts = first_location.str.split(', ').str.len()

# if only 1 part - it's country
subset.loc[n_parts == 1, ['district', 'city']] = pd.NA

# if 2 parts - assume country + district
subset.loc[n_parts == 2, 'city'] = pd.NA

In [25]:
subset[['country', 'district', 'city']].value_counts(dropna=False)

country   district                    city               
Portugal  Lisboa                      Lisboa                 2524
          NaN                         NaN                    2267
          Setúbal                     Barreiro                966
          Região Autónoma da Madeira  Funchal                 569
          Porto                       Porto                   559
                                                             ... 
Japão     NaN                         NaN                       1
Portugal  Santarém                    Mação                     1
          Leiria                      Castanheira de Pera       1
          Santarém                    Cartaxo                   1
          Castelo Branco              Belmonte                  1
Name: count, Length: 308, dtype: int64

In [26]:
subset[subset['district'] == 'Lisboa']['city'].value_counts(dropna=False)

city
Lisboa                    2524
Cascais                    320
Oeiras                     319
Sintra                     215
Loures                     184
<NA>                       149
Torres Vedras               85
Mafra                       78
Amadora                     59
Odivelas                    48
Vila Franca de Xira         41
Alenquer                    21
Lourinhã                    20
Cadaval                     14
Azambuja                     6
Arruda dos Vinhos            4
Sobral de Monte Agraço       3
Name: count, dtype: int64

In [26]:
subset_plot = (
    subset.dropna(subset=["district"])  # remove missing districts
      .groupby("district")
      .agg(
          n_contracts=("idcontrato", "count"),
          n_adjudicantes=("adjudicante", "nunique"),
          n_adjudicatarios=("adjudicatarios", "nunique")
      )
      .reset_index()
)

In [27]:
fig = go.Figure()

# --- traces ---
part01 = subset_plot.sort_values("n_contracts", ascending=False)
fig.add_trace(go.Bar(
    x=part01["district"],
    y=part01["n_contracts"],
    name="Contracts",
    visible=True  # default visible
))

part02 = subset_plot.sort_values("n_adjudicantes", ascending=False)
fig.add_trace(go.Bar(
    x=part02["district"],
    y=part02["n_adjudicantes"],
    name="Adjudicantes",
    visible=False
))

part03 = subset_plot.sort_values("n_adjudicatarios", ascending=False)
fig.add_trace(go.Bar(
    x=part03["district"],
    y=part03["n_adjudicatarios"],
    name="Adjudicatarios",
    visible=False
))

# --- dropdown ---
fig.update_layout(
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label="Contracts",
                    method="update",
                    args=[{"visible": [True, False, False]},
                          {"title": "Number of Contracts"}]
                ),
                dict(
                    label="Adjudicantes",
                    method="update",
                    args=[{"visible": [False, True, False]},
                          {"title": "Number of Adjudicantes"}]
                ),
                dict(
                    label="Adjudicatarios",
                    method="update",
                    args=[{"visible": [False, False, True]},
                          {"title": "Number of Adjudicatarios"}]
                ),
            ],
            direction="down",
            showactive=True
        )
    ]
)

fig.update_layout(
    title="Contracts per District",
    xaxis_title="District",
    yaxis_title="Count",
    xaxis_tickangle=-45
)

fig.show()

In [28]:
#subset = subset[subset['district'] == 'Lisboa']
subset.drop(columns=['LocalExecucao', 'country', 'district'], inplace=True)

In [29]:
subset['city'].value_counts()

city
Lisboa                 2524
Barreiro                966
Funchal                 569
Porto                   559
Braga                   366
                       ... 
Lajes do Pico             1
Mação                     1
Castanheira de Pera       1
Cartaxo                   1
Belmonte                  1
Name: count, Length: 282, dtype: int64

### <font size=6>3.3.6 Price Columns</font> <a class="anchor" id="3.3.6"></a>
  
[Back to TOC](#toc)

In [30]:
# keep idcontrato
price_cols = ['precoBaseProcedimento', 'precoContratual', 'PrecoTotalEfetivo']
box_df = subset[['idcontrato'] + price_cols].copy()

# convert only price columns
box_df[price_cols] = box_df[price_cols].apply(pd.to_numeric, errors='coerce')

# melt while keeping idcontrato
long_df = box_df.melt(
    id_vars='idcontrato',
    var_name='Column',
    value_name='Value'
).dropna()

fig = px.box(
    long_df,
    x='Column',
    y='Value',
    color='Column',
    points='outliers',
    title='Distribution of Price Columns',
    hover_data=['idcontrato'] 
)

fig.update_layout(showlegend=False)
fig.show()

In [31]:
# Count rows with negative contractual price
neg_count = (pd.to_numeric(subset['precoContratual'], errors='coerce') <= 0).sum()
print(f"Rows with precoContratual < 0: {neg_count}")

Rows with precoContratual < 0: 5


In [32]:
# deleting rows where precoContratual is zero or negative, as they are likely errors
subset = subset[subset['precoContratual'] > 0]

In [33]:
subset.info()

<class 'pandas.DataFrame'>
Index: 16989 entries, 2 to 722863
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   idcontrato              16989 non-null  int64         
 1   tipoContrato            16989 non-null  str           
 2   tipoFimContrato         16972 non-null  str           
 3   CPV                     16989 non-null  str           
 4   adjudicante             16989 non-null  str           
 5   adjudicatarios          16989 non-null  str           
 6   concorrentes            14707 non-null  str           
 7   precoBaseProcedimento   16989 non-null  float64       
 8   precoContratual         16989 non-null  float64       
 9   PrecoTotalEfetivo       16989 non-null  float64       
 10  dataDecisaoAdjudicacao  16989 non-null  datetime64[us]
 11  dataCelebracaoContrato  16989 non-null  datetime64[us]
 12  dataPublicacao          16989 non-null  datetime64[us]
 13  d

In [34]:
# Ensure numeric conversion
subset['precoContratual'] = pd.to_numeric(
    subset['precoContratual'],
    errors='coerce'
)

# Filter rows under 50k
under_50k = subset[subset['precoContratual'] < 100]

# Show result
print(f"Rows with precoContratual < 50,000: {len(under_50k)}")

under_50k[
    [
        'idcontrato',
        'precoContratual',
        'precoBaseProcedimento',
        'PrecoTotalEfetivo',
        'adjudicante',
        'adjudicatarios'
    ]
].head(50)

Rows with precoContratual < 50,000: 166


,idcontrato,precoContratual,precoBaseProcedimento,PrecoTotalEfetivo,adjudicante,adjudicatarios
4841,9722477,73.50,46371.10,73.50,503135593 - Administração Regional de Saúde do...,500222665 - Proclinica-Equipamentos e Produtos...
5237,9798956,64.00,104810.00,0.00,506361438 - Instituto Português de Oncologia d...,507958861 - FRILABO II LDA
5862,9717925,96.00,57116.78,96.00,503135593 - Administração Regional de Saúde do...,503731900 - Feiramédica - Importação e Comérci...
10147,9834171,48.64,271350.00,33.44,500745471 - Santa Casa da Misericórdia de Lisboa,"500162166 - LABORATÓRIOS PFIZER, LDA."
10159,9834332,50.60,271350.00,8.28,500745471 - Santa Casa da Misericórdia de Lisboa,"506985261 - SANDOZ FARMACÊUTICA, LDA."
11032,9852472,17.45,21010.17,0.00,680047360 - Serviços de Ação Social da Univers...,500271518 - Sogenave - Sociedade Geral de Abas...
12092,9864374,36.20,96189.65,36.20,503135593 - Administração Regional de Saúde do...,"501895558 - H.R. - HOSPITALAR, LDA"
12094,9864400,95.57,96189.65,95.57,503135593 - Administração Regional de Saúde do...,505348470 - VACUETTE PORTUGAL IMPORT. EXPORT. ...
12506,9868666,87.42,52028.42,78.68,509186998 - CHBM - Centro Hospital Barreiro Mo...,500717419 - SIDEFARMA - SOCIEDADE INDUSTR. EXP...
12539,9869019,82.02,68329.17,52.74,509186998 - Centro Hospitalar Barreiro Montijo...,"500149003 - João Antunes Amaro, Lda"


In [35]:
# Ensure numeric
subset['precoContratual'] = pd.to_numeric(
    subset['precoContratual'],
    errors='coerce'
)

# --- IQR method ---
Q1 = subset['precoContratual'].quantile(0.25)
Q3 = subset['precoContratual'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Find outliers
outliers_df = subset[
    (subset['precoContratual'] < lower_bound) |
    (subset['precoContratual'] > upper_bound)
]

print(f"Number of outliers: {len(outliers_df)}")
print(f"Lower bound: {lower_bound:,.2f}")
print(f"Upper bound: {upper_bound:,.2f}")

# Show relevant columns
outliers_df[
    [
        'idcontrato',
        'precoContratual',
        'precoBaseProcedimento',
        'PrecoTotalEfetivo',
        'adjudicante',
        'adjudicatarios'
    ]
].sort_values('precoContratual', ascending=False)

Number of outliers: 1602
Lower bound: -147,881.55
Upper bound: 271,928.93


,idcontrato,precoContratual,precoBaseProcedimento,PrecoTotalEfetivo,adjudicante,adjudicatarios
637623,12746349,23430000.00,23790000.00,23430000.00,"516192175 - Alsa Todi Metropolitana de Lisboa,...",- - YUTONG FRANCE
651536,13975266,22518600.00,23202500.00,22518600.00,980706840 - Nex Continental Holding SL Sucurs...,- - SAS Yutong France
221225,10535059,13252420.00,23200000.00,13252420.00,600014665 - Secretaria-Geral do Ministério da ...,504615947 - Meo - Serviços de Comunicações e M...
104287,10443075,10213000.00,24200000.00,10213000.00,503135593 - Administração Regional de Saúde do...,"502995912 - Stellantis Portugal, S.A."
188646,10441838,10203968.00,11750000.00,0.00,"505600005 - Águas de Santo André, S. A.","506554791 - ECODEAL, GESTÃO INTEGRAL DE RESÍDU..."
...,...,...,...,...,...,...
370673,11092774,273600.00,406504.06,273600.00,502077352 - Centro de Formação Profissional da...,980476194 - FANUC Iberia S.L.U. - Sucursal em ...
146243,10340524,273600.00,304000.00,15369.01,510345271 - Instituto Nacional de Investigação...,"513589210 - Letras &amp; Pétalas, Unipessoal, ..."
166628,10212933,273370.00,275000.00,273370.00,506792382 - Município de Mealhada,505051931 - AUTO-SUECO PORTUGAL - VEÍCULOS PES...
24453,9981279,272427.32,3655889.57,272073.72,510928897 - Instituto da Segurança Social dos ...,500271518 - Sogenave - Sociedade Geral de Abas...


In [36]:
# TODO: histogram of price distribution with log scale, showing outliers in different color
fig = px.histogram(
    subset,
    x='precoContratual',
    nbins=100,
    title='Distribution of Contractual Price',
    labels={'precoContratual': 'Contractual Price', 'count': 'Number of Contracts'},
    log_y=True
)
fig.show()

### <font size=6>3.3.7 CPV</font> <a class="anchor" id="3.3.7"></a>
  
[Back to TOC](#toc)

In [37]:
#subset['CPV'] = subset['CPV'].astype(str).str.replace('\n', ' | ', regex=True).str.strip()
#subset['CPV'] = subset['CPV'].astype(str).str.replace(r'\s*\d{8}-\d\s*-\s*', ' ', regex=True).str.strip()

In [38]:
subset['CPV'].value_counts() 

CPV
33140000-3 - Material médico de consumo                                                                                                                                   664
45453100-8 - Obras de recuperação                                                                                                                                         409
33190000-8 - Dispositivos e produtos médicos variados                                                                                                                     378
33696500-0 - Reagentes de laboratório                                                                                                                                     302
15800000-6 - Produtos alimentares diversos                                                                                                                                281
                                                                                                                              

In [39]:
subset_counts = subset['CPV'].value_counts().reset_index()
subset_counts.columns = ['CPV', 'count']

# Get top 20
top_20 = subset_counts.head(20)

fig = px.bar(
    top_20,
    x='count',
    y='CPV',
    orientation='h',
    title='Top 20 CPVs by Number of Contracts',
    labels={'CPV': 'CPV', 'count': 'Number of Contracts'},
    width=1700,   
    height=800  
)

# Largest bar on top
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'}
)

fig.show()

In [40]:
# Ensure CPV is string (important)
subset['CPV'] = subset['CPV'].astype(str)

# Extract first 2 digits
subset['cpv_prefix'] = subset['CPV'].str[:2]

# Mapping dictionary (based on your table)
cpv_map = {
    # Primary sectors
    "03": "Agricultura e pesca",
    "77": "Agricultura e pesca",

    "09": "Energia e combustíveis",
    "76": "Energia e combustíveis",

    "14": "Mineração e metais",

    # Manufacturing & goods
    "15": "Alimentação e bebidas",

    "16": "Máquinas e equipamentos",
    "30": "Máquinas e equipamentos",
    "42": "Máquinas e equipamentos",

    "18": "Têxteis e vestuário",
    "19": "Têxteis e vestuário",

    "22": "Material impresso e editorial",

    "24": "Produtos químicos",

    "31": "Equipamento elétrico",
    "32": "Telecomunicações",
    "64": "Telecomunicações",

    "33": "Saúde e equipamento médico",
    "85": "Saúde e equipamento médico",

    "34": "Transportes e veículos",

    "35": "Segurança e defesa",

    "37": "Cultura e desporto",
    "92": "Cultura e desporto",

    "38": "Equipamento científico e laboratorial",

    "39": "Mobiliário e limpeza",

    # Construction
    "43": "Construção",
    "44": "Construção",
    "45": "Construção",

    # IT
    "48": "Tecnologias de Informação",
    "72": "Tecnologias de Informação",

    # Operations
    "50": "Manutenção e reparação",
    "51": "Instalação e montagem",

    # Hospitality
    "55": "Hotelaria e restauração",

    # Transport services
    "60": "Serviços de transporte",
    "63": "Serviços de transporte",

    # Utilities
    "41": "Serviços públicos e utilidades",
    "65": "Serviços públicos e utilidades",

    # Professional services
    "66": "Finanças e seguros",

    "70": "Imobiliário",

    "71": "Arquitetura e engenharia",

    "73": "Investigação e desenvolvimento",

    "79": "Serviços empresariais",

    "80": "Educação",

    "90": "Ambiente e resíduos",

    # Government
    "75": "Administração pública",

    # Other
    "98": "Outros serviços"
}

# Create aggregated CPV category
subset['agg_cpv'] = subset['cpv_prefix'].map(cpv_map).fillna("Outros / Não classificado")

In [41]:
subset_counts = subset['agg_cpv'].value_counts().reset_index()
subset_counts.columns = ['agg_cpv', 'count']

# Get top 20
top_20 = subset_counts.head(20)

fig = px.bar(
    top_20,
    x='count',
    y='agg_cpv',
    orientation='h',
    title='Top 20 CPVs by Number of Contracts',
    labels={'agg_cpv': 'agg_cpv', 'count': 'Number of Contracts'},
    width=1700,   
    height=800  
)

# Largest bar on top
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'}
)

fig.show()

### <font size=6>3.3.8 Adjudicante & Adjudicatário</font> <a class="anchor" id="3.3.8"></a>
  
[Back to TOC](#toc)

In [42]:
# extract contribuinte numbers from adjudicante and adjudicatarios
subset['contribuinte_adjudicante'] = subset['adjudicante'].str.extract(r'(\d{9})')
subset['adjudicante'] = subset['adjudicante'].str.replace(r'\s*\d{9}\s* - ', '', regex=True).str.strip()

subset['contribuinte_adjudicatarios'] = subset['adjudicatarios'].str.extract(r'(\d{9})')
subset['adjudicatarios'] = subset['adjudicatarios'].str.replace(r'\s*\d{9}\s* - ', '', regex=True).str.strip()

In [43]:
# Detect names starting with dashes
mask = (
    subset['adjudicante'].astype(str).str.strip().str.startswith('-')
    |
    subset['adjudicatarios'].astype(str).str.strip().str.startswith('-')
)

bad_rows = subset[mask]

print(f"Rows with malformed entity names: {len(bad_rows)}")

bad_rows[
    [
        'idcontrato',
        'adjudicante',
        'adjudicatarios'
    ]
].head(10)

Rows with malformed entity names: 434


,idcontrato,adjudicante,adjudicatarios
842,9697397,Município do Porto,- - ANA ISABEL TEIXEIRA FERREIRA
1829,9701917,Instituto Superior de Economia e Gestão da Uni...,- - EBSCO INFORMATION SERVICES S.L.O.
2816,9726823,Instituto Superior de Economia e Gestão,- - Proquest LLC
6401,9810361,Município de Marco de Canaveses,"- - MONDO PORTUGAL, S.A."
7998,9802243,Instituto Politécnico de Viana do Castelo,- - Sound of Numbers SL
10487,9838581,Serviço de Saúde da Região Autónoma da Madeira...,"- - ISÓTOPOS E DERIVADOS (ISODER), S.A."
12328,9866602,"CHBM - Centro Hospital Barreiro Montijo, EPE","- - LABESFAL - LABORATÓRIOS ALMIRO, S.A."
12800,9871639,"CHBM - Centro Hospital Barreiro Montijo, EPE","- - LABORATÓRIOS URGO, S. L"
14490,9824583,Escola Básica Integrada de Canto da Maia,- - UNISELF SA
14565,9894124,"Casa Pia de Lisboa, I. P.",- - Antonio Rogerio Cabral Rodrigues Canhoes


In [44]:
## LEGAL VARIANTS

adg = subset['adjudicante'].astype(str)
adj = subset['adjudicatarios'].astype(str)

all_entities = pd.concat([adg, adj], ignore_index=True)

legal_df = pd.DataFrame({"original": all_entities})
legal_df["clean"] = legal_df["original"].str.lower().str.strip()

legal_patterns = [
    r'\bs\.?a\.?\b',
    r'\bsociedade anonima\b',
    r'\bltda\b',
    r'\bunipessoal ltda\b',
    r'\bs\.?l\.?\b',
    r'\bs\.?a\.?u\.?\b',
    r'\bs\.?l\.?u\.?\b',
    r'\binc\b',
    r'\bllc\b'
]

for pat in legal_patterns:
    legal_df["base"] = legal_df["clean"].str.replace(pat, "", regex=True)

legal_df["base"] = (
    legal_df["base"]
    .str.replace(r'[.,;:()\-]', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

legal_groups = (
    legal_df
    .groupby("base")
    .agg(
        n_variants=("original", "nunique"),
        examples=("original", lambda x: list(set(x)))
    )
    .reset_index()
)

legal_duplicates = legal_groups[legal_groups["n_variants"] > 1]

print(f"Companies differing only by legal form: {len(legal_duplicates)}")

legal_dup_df = legal_duplicates.sort_values("n_variants", ascending=False)

legal_dup_df.head(50)

Companies differing only by legal form: 1112


,base,n_variants,examples
4332,meo serviços de comunicações e multimédia s a,12,"[MEO - Serviços de Comunicações e Multimédia, ..."
6403,suma serviços urbanos e meio ambiente s a,9,"[Suma - Serviços Urbanos e Meio Ambiente, S.A...."
800,b braun medical unipessoal lda,8,"[B. Braun Medical, Unipessoal Lda., B.Braun Me..."
2479,exitus soluções tecnológicas lda,8,"[Exitus, Soluções Tecnológicas, Lda, EXITUS, S..."
1441,claranet ii solutions s a,8,"[CLARANET II SOLUTIONS, S.A,, - - CLARANET II ..."
3318,inetum españa s a sucursal em portugal,7,"[INETUM ESPAÑA S.A. - SUCURSAL EM PORTUGAL, In..."
847,basedois informática e telecomunicações lda,7,"[Basedois - Informática E Telecomunicações, Ld..."
2596,fidelidade companhia de seguros s a,7,"[Fidelidade - Companhia de Seguros, S.A., FIDE..."
5431,prn informática lda,7,"[PRN Informática, Lda.,, PRN INFORMÁTICA, LDA,..."
6345,spormex events &amp exhibitions lda,7,"[SPORMEX - EVENTS &amp; EXHIBITIONS, LDA, SPOR..."


In [45]:
def clean_entity(x):
    x = str(x)

    # remove accents
    x = unicodedata.normalize('NFKD', x).encode('ASCII', 'ignore').decode('utf-8')

    # lowercase
    x = x.lower()

    # remove punctuation
    x = re.sub(r'[.,;:()\-]', ' ', x)

    # normalize spaces
    x = re.sub(r'\s+', ' ', x).strip()

    # -------------------------
    # REMOVE LEGAL FORMS
    # -------------------------

    legal_patterns = [
        r'\bsa\b',
        r'\bs a\b',
        r'\bs a u\b',
        r'\bsau\b',
        r'\bs l\b',
        r'\bsl\b',
        r'\bs l u\b',
        r'\bsociedade anonima\b',
        r'\bsoc anonima\b',
        r'\bltda\b',
        r'\bunipessoal ltda\b',
        r'\bunipessoal lda\b',
        r'\blda\b',
        r'\binc\b',
        r'\bllc\b'
    ]

    for pat in legal_patterns:
        x = re.sub(pat, '', x)

    # final cleanup after removals
    x = re.sub(r'\s+', ' ', x).strip()

    return x

subset['adjudicante_clean'] = subset['adjudicante'].apply(clean_entity)
subset['adjudicatarios_clean'] = subset['adjudicatarios'].apply(clean_entity)

In [46]:
## LEGAL VARIANTS

adg = subset['adjudicante_clean'].astype(str)
adj = subset['adjudicatarios_clean'].astype(str)

all_entities = pd.concat([adg, adj], ignore_index=True)

legal_df = pd.DataFrame({"original": all_entities})
legal_df["clean"] = legal_df["original"].str.lower().str.strip()

legal_patterns = [
    r'\bs\.?a\.?\b',
    r'\bsociedade anonima\b',
    r'\bltda\b',
    r'\bunipessoal ltda\b',
    r'\bs\.?l\.?\b',
    r'\bs\.?a\.?u\.?\b',
    r'\bs\.?l\.?u\.?\b',
    r'\binc\b',
    r'\bllc\b'
]

for pat in legal_patterns:
    legal_df["base"] = legal_df["clean"].str.replace(pat, "", regex=True)

legal_df["base"] = (
    legal_df["base"]
    .str.replace(r'[.,;:()\-]', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

legal_groups = (
    legal_df
    .groupby("base")
    .agg(
        n_variants=("original", "nunique"),
        examples=("original", lambda x: list(set(x)))
    )
    .reset_index()
)

legal_duplicates = legal_groups[legal_groups["n_variants"] > 1]

print(f"Companies differing only by legal form: {len(legal_duplicates)}")

legal_dup_df = legal_duplicates.sort_values("n_variants", ascending=False)

legal_dup_df.head(50)

Companies differing only by legal form: 0


,base,n_variants,examples


## <font size=6>**3.4 Final Data**</font> <a class="anchor" id="3.4"></a>
  
[Back to TOC](#toc)

In [47]:
subset.head(15)

,idcontrato,tipoContrato,tipoFimContrato,CPV,adjudicante,adjudicatarios,concorrentes,precoBaseProcedimento,precoContratual,PrecoTotalEfetivo,...,dataPublicacao,dataFechoContrato,nr_concorrentes,city,cpv_prefix,agg_cpv,contribuinte_adjudicante,contribuinte_adjudicatarios,adjudicante_clean,adjudicatarios_clean
2,9664960,Aquisição de bens móveis,Anulado ou Declarado Nulo,30213100-6 - Computadores portáteis,Universidade do Algarve,"Empis - Informática e Serviços, Lda",501333401-BASE2 - Informática e Telecomunicaçõ...,320620.41,3260.00,0.00,...,2023-01-02,2023-12-31,15,Faro,30,Máquinas e equipamentos,505387271,502163518,universidade do algarve,empis informatica e servicos
24,9668634,Aquisição de bens móveis,Resolução do Contrato,09134100-8 - Gasóleo,Município de Góis,Alves Bandeira SA,500433402-Alves Bandeira SA,374112.00,327326.40,343994.23,...,2023-01-03,2026-01-16,1,Góis,09,Energia e combustíveis,506613399,500433402,municipio de gois,alves bandeira
30,9664045,Aquisição de bens móveis,"O cumprimento, a impossibilidade definitiva e ...",15811000-6 - Produtos de panificação,Serviços de Ação Social da Universidade do Minho,Padaria Trinas lda,500209634-Padaria Trinas lda,85424.72,84659.05,84659.05,...,2023-01-02,2023-08-22,1,Braga,15,Alimentação e bebidas,680047360,500209634,servicos de acao social da universidade do minho,padaria trinas
33,9458872,Empreitadas de obras públicas,"O cumprimento, a impossibilidade definitiva e ...",45112700-2 - Trabalhos de paisagismo,Município de Barcelos,"M. Couto Alves, S.A.","500553408-ALEXANDRE BARBOSA BORGES, S.A. | 500...",3615610.00,3519310.80,3429226.77,...,2022-09-20,2025-10-10,13,Barcelos,45,Construção,505584760,504213709,municipio de barcelos,m couto alves
71,9674355,Empreitadas de obras públicas,"O cumprimento, a impossibilidade definitiva e ...",45453100-8 - Obras de recuperação,Gebalis - Gestão do Arrendamento da Habitação ...,Metangular Construções Lda.,"509944647-Construbuild - Services, Limitada | ...",362000.00,57393.03,57393.03,...,2023-01-04,2023-06-29,13,Lisboa,45,Construção,503541567,510826768,gebalis gestao do arrendamento da habitacao mu...,metangular construcoes
80,9674848,Aquisição de bens móveis,"O cumprimento, a impossibilidade definitiva e ...",39298700-4 - Troféus,Município de Vila Nova de Famalicão,"Gravymedal, Comércio e Personalização de Prémi...","510128211-Ana Maria Araújo Oliveira e Silva, U...",40000.00,1150.00,1150.00,...,2023-01-04,2023-04-19,2,Vila Nova de Famalicão,39,Mobiliário e limpeza,506663264,507232283,municipio de vila nova de famalicao,gravymedal comercio e personalizacao de premios
96,9676561,Aquisição de bens móveis,"O cumprimento, a impossibilidade definitiva e ...",44164000-7 - Tubagem de revestimento e tubos,SIMAR – Serviços Intermunicipalizados de Loure...,"HUMBERTO POÇAS, S.A.",NaN,72500.00,519.00,519.00,...,2023-01-05,2023-10-04,0,Loures,44,Construção,680009671,501075666,simar servicos intermunicipalizados de loures ...,humberto pocas
104,9668212,Aquisição de bens móveis,"O cumprimento, a impossibilidade definitiva e ...",15111000-9 - Carne de bovino,Serviços de Ação Social da Universidade do Minho,Carnes S. José - Comércio e Indústria de Carne...,509061800-Consumintenso - Produtos Alimentares...,92435.50,55952.50,47148.20,...,2023-01-03,2023-07-21,5,Braga,15,Alimentação e bebidas,680047360,501235574,servicos de acao social da universidade do minho,carnes s jose comercio e industria de carnes
109,9677208,Aquisição de bens móveis | Aquisição de serviços,Revogação do Contrato,15894200-3 - Refeições preparadas\r\n55322000-...,"A. R. M. - Águas e Resíduos da Madeira, S. A.",Gertal-Companhia Geral de Restaurantes e Alime...,500126623-GERTAL - Companhia Geral de Restaura...,174300.00,174281.40,70001.12,...,2023-01-05,2023-11-01,2,Santa Cruz,15,Alimentação e bebidas,509574513,500126623,a r m aguas e residuos da madeira,gertal companhia geral de restaurantes e alime...
111,9668443,Aquisição de bens móveis,"O cumprimento, a impossibilidade definitiva e ..

In [48]:
subset.info()

<class 'pandas.DataFrame'>
Index: 16989 entries, 2 to 722863
Data columns (total 22 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   idcontrato                   16989 non-null  int64         
 1   tipoContrato                 16989 non-null  str           
 2   tipoFimContrato              16972 non-null  str           
 3   CPV                          16989 non-null  str           
 4   adjudicante                  16989 non-null  str           
 5   adjudicatarios               16989 non-null  str           
 6   concorrentes                 14707 non-null  str           
 7   precoBaseProcedimento        16989 non-null  float64       
 8   precoContratual              16989 non-null  float64       
 9   PrecoTotalEfetivo            16989 non-null  float64       
 10  dataDecisaoAdjudicacao       16989 non-null  datetime64[us]
 11  dataCelebracaoContrato       16989 non-null  datetime64[

In [49]:
print("There are {} Public Entities.".format(subset['adjudicante_clean'].nunique()))
print("There are {} Companies.".format(subset['adjudicatarios_clean'].nunique()))

print("So, in total our analysis contains {} Nodes.".format(
    subset['adjudicatarios_clean'].nunique() + 
    subset['adjudicante_clean'].nunique()))

There are 1109 Public Entities.
There are 5338 Companies.
So, in total our analysis contains 6447 Nodes.


In [50]:
print("Initial Number of Contracts: {}".format(merged_dataset.shape[0]))
print("Number of Contracts Now: {}".format(subset.shape[0]))
print("Percentage of Deleted Contracts: {}%".format(round((1 - (subset.shape[0]/merged_dataset.shape[0]))*100, 2)))

Initial Number of Contracts: 732573
Number of Contracts Now: 16989
Percentage of Deleted Contracts: 97.68%


# <font color='#BFD72F' size=6>**4. Export Preprocessed Data**</font> <a class="anchor" id="4"></a>
  
[Back to TOC](#toc)

In [51]:
subset.to_csv("../data/preprocessed_data.csv", index=False)